# Case Study §3 — Continued Pretraining (CPT) on the chemistry corpus

Runnable twin of [`03_cpt.py`](03_cpt.py). CPT = next-token prediction over raw domain text →
**full causal loss on every token** (we print the unmasked fraction to *prove* it is ~100%). We then
measure **held-out perplexity before vs after** to show the model absorbed the domain.

### Two things this section documents
1. **Packing** — how documents are laid into fixed-length sequences. HF backend uses *manual concatenation*
   packing (docs glued with EOS, cut into `block_size` chunks); Unsloth uses `packing=True` (best-fit,
   padding-free). See `RESEARCH_NOTES.md` §8–9.
2. **HF vs Unsloth** — same CPT, two backends. Unsloth *fully* trains embeddings (more trainable params)
   and auto-applies a smaller `embedding_learning_rate`; HF LoRA-adapts them and we set the split LR by hand.

> **Top-level flags** below. `MODE="trial"` runs in seconds to validate plumbing; set `"full"` for the real run.
> `BACKEND="hf"` works in this (`.venv-rl`) kernel; `"unsloth"` requires the `.venv` kernel.

In [ ]:
# ── TOP-LEVEL FLAGS ───────────────────────────────────────────────
MODE = "trial"      # "trial" (fast smoke test) or "full" (real run)
BACKEND = "hf"      # "hf" (this kernel) or "unsloth" (run with the .venv kernel)
FORCE = False        # retrain even if a cached adapter exists
# ──────────────────────────────────────────────────────────────────
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = HERE if (HERE / "03_cpt.py").exists() else HERE / "case_study"
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("cpt", root / "03_cpt.py")
cpt = importlib.util.module_from_spec(spec); spec.loader.exec_module(cpt)
print(f"mode={config.RUN_MODE} backend={BACKEND} limits={config.limits()}")

## Packing, demonstrated
`pack_blocks` concatenates documents (EOS-separated) and chunks them into fixed `block_size` blocks.
This is *manual concatenation packing* — efficient, standard for CPT. (Unsloth's `packing=True` is the
best-fit variant that also avoids cross-document attention.)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(config.BASE_MODEL)
demo = cpt.pack_blocks(tok, ["Density functional theory is a method.", "Hartree-Fock uses a single determinant."], block_size=16)
print(f"2 short docs -> {len(demo)} block(s) of 16 tokens; EOS id {tok.eos_token_id} separates documents")
print('first block ids:', demo[0])
assert all(len(b) == 16 for b in demo), 'blocks are fixed-length'

## Run CPT
Self-sufficient (builds the corpus if missing, downloads SmolLM2-135M on first use) and idempotent
(re-running loads the cached adapter+metrics unless `FORCE=True`).

In [ ]:
metrics = cpt.run_cpt(backend=BACKEND, force=FORCE)
print(json.dumps({k: metrics[k] for k in ['backend','packing','ppl_before','ppl_after','ppl_delta_pct'] if k in metrics}, indent=2))

## Verify
Assert the section's claims hold: full causal loss (HF backend reports 100% unmasked), perplexity was
measured before and after, and an adapter was saved.

In [ ]:
assert 'ppl_before' in metrics and 'ppl_after' in metrics, 'perplexity must be measured'
if metrics.get('backend') == 'hf':
    assert metrics['unmasked_fraction'] == 1.0, 'CPT must use FULL causal loss (0% masked)'
adapter = cpt._out_dir(metrics['backend']) / 'adapter'
assert adapter.exists(), 'adapter should be saved'
print(f"\u2713 {metrics['backend']} CPT verified: ppl {metrics['ppl_before']:.2f} -> {metrics['ppl_after']:.2f}"
      f" ({metrics['ppl_delta_pct']:+.1f}%), adapter at {adapter}")
print('\nNote: in TRIAL the perplexity barely moves (only a few steps) \u2014 that is expected. Set MODE=\"full\"')
print('for the real drop. Next: \u00a74 base-vs-instruct + catastrophic-forgetting check.')